In [ ]:
# the smallest possible baseline code for the meta-pool project


import random

from typing import Tuple
import matplotlib.pyplot as plt
import numpy as np


class CollaborativeLearningGroup:
    """A group of learners collaborating toward a common goal."""
    def __init__(self, learners):
        self.learners = learners

    def collaborate(self):
        """Allow learners to share experiences and reinforce learning."""
        for learner in self.learners:
            for peer in self.learners:
                if learner != peer:
                    learner.q_table += 0.1 * (peer.q_table - learner.q_table)


class DynaQAgent:
    """DynaQAgent with collaborative learning.

    Attributes:
    ----------
    n_states (int): Number of states in the environment.
    n_actions (int): Number of possible actions in the environment.
    epsilon (float): Probability of choosing a random action (exploration).
    alpha (float): Learning rate for updating Q-values.
    gamma (float): Discount factor for future rewards.
    planning_steps (int): Number of planning steps to perform.
    q_table (ndarray): Q-values table for state-action pairs.
    model (dict): Model to store state transitions and rewards.
    """

    def __init__(
        self,
        n_states: int,
        n_actions: int,
        epsilon: float = 0.1,
        alpha: float = 0.1,
        gamma: float = 0.95,
        planning_steps: int = 5,
    ) -> None:
        self.n_states = n_states
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}

    def choose_action(self, state: int) -> int:
        if random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.n_actions)
        return np.argmax(self.q_table[state])

    def update(self, state: int, action: int, reward: float, next_state: int) -> None:
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error
        self.model[(state, action)] = (reward, next_state)

        for _ in range(self.planning_steps):
            s, a = random.choice(list(self.model.keys()))
            r, s_next = self.model[(s, a)]
            best_next_a = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_a]
            td_error = td_target - self.q_table[s][a]
            self.q_table[s][a] += self.alpha * td_error


def simulate_collaborative_learning(agents, teacher, episodes=100):
    rewards = []
    for episode in range(episodes):
        for agent in agents:
            state = random.randint(0, agent.n_states - 1)
            action = agent.choose_action(state)
            reward = teacher.evaluate(state, action)
            next_state = (state + 1) % agent.n_states
            agent.update(state, action, reward, next_state)
        CollaborativeLearningGroup(agents).collaborate()
        rewards.append(np.mean([np.max(agent.q_table) for agent in agents]))
    return rewards

# Example usage
num_agents = 3
agents = [DynaQAgent(n_states=10, n_actions=4) for _ in range(num_agents)]
teacher = DynaQAgent(n_states=10, n_actions=4)
avg_rewards = simulate_collaborative_learning(agents, teacher)
print("Collaborative learning completed.")


In [ ]:
##version 2
import random

from typing import Tuple
import matplotlib.pyplot as plt
import numpy as np


class CollaborativeLearningGroup:
    """A group of learners collaborating toward a common goal."""
    def __init__(self, learners):
        self.learners = learners

    def add_learner(self, new_learner):
        """Add a new learner to the group."""
        self.learners.append(new_learner)

    def collaborate(self):
        """Allow learners to share experiences and reinforce learning."""
        for learner in self.learners:
            for peer in self.learners:
                if learner != peer:
                    learner.q_table += 0.1 * (peer.q_table - learner.q_table)


class DynaQAgent:
    """DynaQAgent with collaborative learning and teacher intervention."""
    def __init__(
        self,
        n_states: int,
        n_actions: int,
        epsilon: float = 0.1,
        alpha: float = 0.1,
        gamma: float = 0.95,
        planning_steps: int = 5,
    ) -> None:
        self.n_states = n_states
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}

    def choose_action(self, state: int) -> int:
        if random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.n_actions)
        return np.argmax(self.q_table[state])

    def update(self, state: int, action: int, reward: float, next_state: int) -> None:
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error
        self.model[(state, action)] = (reward, next_state)

        for _ in range(self.planning_steps):
            s, a = random.choice(list(self.model.keys()))
            r, s_next = self.model[(s, a)]
            best_next_a = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_a]
            td_error = td_target - self.q_table[s][a]
            self.q_table[s][a] += self.alpha * td_error


class SimulatedTeacher:
    """A class to simulate a teacher guiding a human learner."""
    def __init__(self, guidance_threshold=0.5):
        self.guidance_threshold = guidance_threshold

    def evaluate(self, state, action):
        return 1 if random.uniform(0, 1) > 0.2 else 0  # Simulating rewards

    def should_intervene(self, avg_reward: float) -> bool:
        return np.random.binomial(1, 1 - avg_reward) == 1

    def guide(self, learner):
        learner.q_table += 0.1 * np.random.rand(*learner.q_table.shape)


def simulate_collaborative_learning(agents, teacher, teacher_intervention=True, episodes=100):
    rewards = []
    group = CollaborativeLearningGroup(agents)
    
    for episode in range(episodes):
        for agent in agents:
            state = random.randint(0, agent.n_states - 1)
            action = agent.choose_action(state)
            reward = teacher.evaluate(state, action)
            next_state = (state + 1) % agent.n_states
            agent.update(state, action, reward, next_state)

            if teacher_intervention and teacher.should_intervene(np.mean(rewards[-10:]) if len(rewards) >= 10 else 1):
                teacher.guide(agent)
        
        group.collaborate()
        rewards.append(np.mean([np.max(agent.q_table) for agent in agents]))
    return rewards


def add_new_student(group, n_states=10, n_actions=4):
    """Interface to add a new student to the group dynamically."""
    new_student = DynaQAgent(n_states, n_actions)
    group.add_learner(new_student)
    print("New student added to the group.")


# Example usage
num_agents = 3
agents = [DynaQAgent(n_states=10, n_actions=4) for _ in range(num_agents)]
teacher = SimulatedTeacher(guidance_threshold=0.5)
avg_rewards = simulate_collaborative_learning(agents, teacher, teacher_intervention=True)
print("Collaborative learning with teacher intervention completed.")

# Add a new student dynamically
group = CollaborativeLearningGroup(agents)
add_new_student(group)


In [ ]:
class Environment:
    """Simulated learning environment with continuous feedback."""
    def __init__(self, n_states: int, goal_state: int):
        self.n_states = n_states
        self.goal_state = goal_state

    def step(self, state: int, action: int, peers_q_tables) -> Tuple[int, float]:
        """Defines state transitions and assigns realistic rewards."""
        next_state = (state + action) % self.n_states  # Example transition
        
        # Reward for correct action
        base_reward = 0.5 if next_state > state else -0.1  # Small reward for progress

        # Reward for learning from peers (collaborative reinforcement)
        peer_influence = np.mean([q_table[state, action] for q_table in peers_q_tables])
        social_reward = 0.2 * peer_influence  # Encourages knowledge sharing

        total_reward = base_reward + social_reward
        return next_state, total_reward
